# Notebook 5: GraphSAGE — Inductive Learning

GCN and GAT are **transductive**: they must see all nodes during training. What if you need to classify new nodes that appear after training — new users on a platform, new proteins in a database?

**GraphSAGE** (Hamilton et al., 2017) solves this with **inductive** learning: it learns an *aggregator function* over sampled neighborhoods, not embeddings for specific nodes.

**Prerequisites:** Notebooks 1–4

**What you'll learn:**
- Transductive vs. inductive learning (deep dive)
- GraphSAGE algorithm: sample, aggregate, combine
- Different aggregators: mean, pooling, LSTM
- Mini-batch training with `NeighborLoader`
- Using PyG's `SAGEConv`

**By the end:** train GraphSAGE with neighborhood sampling on a large graph.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.datasets import Planetoid, Reddit2
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader

torch.manual_seed(42)
np.random.seed(42)

# We'll use Cora for comparisons, then show scaling with Reddit
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
print(f'Cora: {data.num_nodes} nodes, {data.num_edges} edges')

---
## 1. Transductive vs. Inductive — What's the Difference?

| Property | Transductive (GCN, GAT) | Inductive (GraphSAGE) |
|----------|------------------------|---------------------|
| **Training nodes** | All nodes seen | Subset of nodes |
| **At test time** | Same graph, new labels | Brand new nodes/graphs |
| **Embeddings** | Per-node lookup table | Computed by aggregating neighbors |
| **Memory** | O(N) for all embeddings | O(batch size) |
| **Use case** | Fixed knowledge graph | Evolving networks, new molecules |

**Why does this matter?**
- GCN optimizes embeddings for *specific* node indices during training
- GraphSAGE optimizes *how to aggregate* from any neighborhood
- At inference time, apply the same aggregation to new nodes → generalizes

---
## 2. The GraphSAGE Algorithm

For each node $v$ in the current batch:

```
1. SAMPLE: randomly sample k neighbors from N(v)
2. AGGREGATE: h_N(v) = AGGREGATE({h_u : u in SAMPLE(N(v))})
3. COMBINE: h_v_new = sigma(W * CONCAT(h_v, h_N(v)))
4. NORMALIZE: h_v_new = h_v_new / ||h_v_new||_2
```

**Key:** The aggregator $\phi$ is shared across all nodes. Learn $\phi$, not per-node embeddings.

In [ ]:
# ============================================================
# GraphSAGE Layer from scratch — showing the mean aggregator
# ============================================================
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops

class SAGEConvScratch(MessagePassing):
    """
    GraphSAGE with mean aggregator, from scratch.
    
    The key difference from GCN:
    - We CONCAT (not add) own features with aggregated neighbor features
    - This allows the model to distinguish a node from its neighborhood
    - No symmetric normalization — just mean over neighbors
    """
    
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='mean')  # mean aggregation over neighbors
        # W_self: transforms own features
        # W_neigh: transforms aggregated neighbor features
        # Together they form the 'combine' step
        self.lin_self = nn.Linear(in_channels, out_channels, bias=False)
        self.lin_neigh = nn.Linear(in_channels, out_channels, bias=False)
        nn.init.xavier_uniform_(self.lin_self.weight)
        nn.init.xavier_uniform_(self.lin_neigh.weight)
    
    def forward(self, x, edge_index):
        # Separate: transform own features
        out_self = self.lin_self(x)
        # Aggregate and transform neighbor features
        out_neigh = self.propagate(edge_index, x=x)
        # Combine: sum (equivalent to concat + one big linear)
        return out_self + out_neigh
    
    def message(self, x_j):
        # x_j: source node features (neighbors)
        # aggr='mean' will average these at the destination
        return self.lin_neigh(x_j)


# Test
conv_sage_scratch = SAGEConvScratch(data.num_features, 64)
out_scratch = conv_sage_scratch(data.x, data.edge_index)
print(f'SAGEConv scratch output: {out_scratch.shape}')

In [ ]:
# ============================================================
# GraphSAGE model using PyG's SAGEConv
# ============================================================

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_layers=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.convs.append(SAGEConv(hidden_channels, out_channels))
    
    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


torch.manual_seed(42)
model_sage = GraphSAGE(
    in_channels=dataset.num_features,
    hidden_channels=256,
    out_channels=dataset.num_classes,
    num_layers=2
)
print(model_sage)
print(f'Params: {sum(p.numel() for p in model_sage.parameters()):,}')

---
## 3. Full-Graph Training (transductive mode)

First, let's train GraphSAGE on the full Cora graph — same as we did with GCN/GAT — so we can compare numbers.

In [ ]:
optimizer_sage = torch.optim.Adam(model_sage.parameters(), lr=0.01)

best_val, best_test = 0, 0

for epoch in range(1, 201):
    model_sage.train()
    optimizer_sage.zero_grad()
    out = model_sage(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer_sage.step()
    
    model_sage.eval()
    with torch.no_grad():
        pred = model_sage(data.x, data.edge_index).argmax(1)
    val = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
    test = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    if val > best_val:
        best_val, best_test = val, test
    
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | Val: {val:.4f} | Test: {test:.4f}')

print(f'\nBest Val: {best_val:.4f} | Test at best: {best_test:.4f}')

---
## 4. Mini-Batch Training with NeighborLoader

For large graphs (Reddit: 232K nodes, 114M edges), loading the full graph is impossible. **NeighborLoader** samples a mini-batch:

1. Start with a batch of seed nodes
2. Sample `num_neighbors` neighbors for each node at each layer depth
3. Build a small subgraph — only what's needed for this batch
4. Forward pass on the subgraph

This is what makes GraphSAGE practical at scale.

```
Seed nodes: [42, 107, 891]
  Layer 2 sampling: each seed samples 10 neighbors -> ~30 nodes
  Layer 1 sampling: each of those samples 10 -> ~300 nodes
Total: ~333 nodes instead of 232K
```

In [ ]:
# NeighborLoader on Cora (tiny graph, but illustrates the concept)
train_loader = NeighborLoader(
    data,
    num_neighbors=[10, 10],   # sample 10 neighbors at hop 2, 10 at hop 1
    batch_size=64,
    input_nodes=data.train_mask,
    shuffle=True,
)

val_loader = NeighborLoader(
    data,
    num_neighbors=[10, 10],
    batch_size=64,
    input_nodes=data.val_mask,
)

# Inspect one batch
batch = next(iter(train_loader))
print('=== Mini-batch Structure ===')
print(f'Batch nodes (seed + neighbors): {batch.num_nodes}')
print(f'Batch edges: {batch.num_edges}')
print(f'Seed nodes in this batch: {batch.batch_size}')  # first N nodes are seeds
print(f'batch.x shape: {batch.x.shape}')
print(f'batch.y shape: {batch.y.shape}')
print()
print('Note: predictions are only needed for the SEED nodes ([:batch.batch_size])')
print('Neighbor nodes are included only to provide context for message passing.')

In [ ]:
# GraphSAGE model for mini-batch training
# Same model architecture, different training loop!

torch.manual_seed(42)
model_batch = GraphSAGE(
    in_channels=dataset.num_features,
    hidden_channels=256,
    out_channels=dataset.num_classes
)
optimizer_batch = torch.optim.Adam(model_batch.parameters(), lr=0.01)


def train_minibatch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        # CRITICAL: only compute loss on seed nodes (first batch_size nodes)
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_minibatch(model, loader):
    model.eval()
    total_correct = 0
    total_nodes = 0
    for batch in loader:
        out = model(batch.x, batch.edge_index)
        pred = out[:batch.batch_size].argmax(1)
        total_correct += (pred == batch.y[:batch.batch_size]).sum().item()
        total_nodes += batch.batch_size
    return total_correct / total_nodes


best_val, best_test = 0, 0

for epoch in range(1, 21):
    loss = train_minibatch(model_batch, train_loader, optimizer_batch)
    val_acc = eval_minibatch(model_batch, val_loader)
    
    # Full-graph test eval
    model_batch.eval()
    with torch.no_grad():
        pred_full = model_batch(data.x, data.edge_index).argmax(1)
    test_acc = (pred_full[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    
    if val_acc > best_val:
        best_val, best_test = val_acc, test_acc
    
    print(f'Epoch {epoch:>2} | Loss: {loss:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}')

print(f'\nBest Val: {best_val:.4f} | Test: {best_test:.4f}')

---
## 5. Aggregator Variants

GraphSAGE paper tests 3 aggregators:

| Aggregator | Formula | Properties |
|-----------|---------|----------|
| **Mean** | avg of neighbor features | Simple, fast, good baseline |
| **Max-Pool** | max(MLP(h_u)) over neighbors | Captures max response per feature |
| **LSTM** | LSTM over *random permutation* | Most expressive, but order-dependent |

PyG's `SAGEConv` uses mean aggregation. For max-pool, use the `aggr='max'` argument.

In [ ]:
# Compare mean vs max-pool aggregation

class SAGEVariant(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, aggr='mean', dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr=aggr)
        self.conv2 = SAGEConv(hidden_channels, out_channels, aggr=aggr)
    
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


results = {}
for aggr in ['mean', 'max', 'sum']:
    torch.manual_seed(42)
    m = SAGEVariant(dataset.num_features, 256, dataset.num_classes, aggr=aggr)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    
    best_v, best_t = 0, 0
    for epoch in range(200):
        m.train(); opt.zero_grad()
        out = m(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            pred = m(data.x, data.edge_index).argmax(1)
        v = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
        t = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        if v > best_v: best_v, best_t = v, t
    
    results[aggr] = (best_v, best_t)
    print(f'SAGEConv (aggr={aggr:<4}) | Val: {best_v:.4f} | Test: {best_t:.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
aggrs = list(results.keys())
test_accs = [results[a][1] for a in aggrs]
bars = ax.bar(aggrs, test_accs, color=['steelblue', 'coral', 'mediumseagreen'])
for bar, acc in zip(bars, test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', fontsize=11)
ax.set_ylim(0.7, 0.88)
ax.set_xlabel('Aggregation type')
ax.set_ylabel('Test accuracy (Cora)')
ax.set_title('GraphSAGE: Effect of aggregation function')
plt.tight_layout()
plt.show()

---
# MINI-PROJECT 5: GraphSAGE Mini-Batch on Amazon Dataset

**Task:** Train GraphSAGE with mini-batch sampling on the Amazon-Computers or Amazon-Photo co-purchase graph — a much larger graph than Cora.

**Requirements:**
1. Load `Amazon` dataset (`Computers` or `Photo`) from PyG
2. Manually create train/val/test masks (20 nodes per class for train, rest split 85/15 val/test)
3. Set up `NeighborLoader` with appropriate `num_neighbors` and `batch_size`
4. Train GraphSAGE with mini-batch training for 10-20 epochs
5. Evaluate on the full graph at the end
6. Plot training loss and validation accuracy per epoch
7. **Ablation:** try `num_neighbors=[5,5]` vs `[15,10]` vs `[25,15]` — what's the accuracy-speed tradeoff?

**Hints:** Amazon-Computers has 13,752 nodes and 491,722 edges — significantly larger than Cora.

In [ ]:
# MINI-PROJECT SKELETON
from torch_geometric.datasets import Amazon
from torch_geometric.loader import NeighborLoader
import torch

# TODO 1: Load Amazon dataset
# dataset = Amazon(root='/tmp/Amazon', name='Computers')
# data = dataset[0]
# print(data)

# TODO 2: Create train/val/test masks
# 20 nodes per class for training (similar to Cora)
# Hint:
def create_masks(data, train_per_class=20, val_ratio=0.15):
    """
    Returns train_mask, val_mask, test_mask tensors.
    20 nodes per class for training, val_ratio of remaining for val, rest for test.
    """
    num_nodes = data.num_nodes
    num_classes = data.y.max().item() + 1
    
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    
    # TODO: for each class, randomly pick train_per_class nodes
    # Remaining nodes: split val_ratio to val, rest to test
    
    return train_mask, val_mask, test_mask


# TODO 3: Create NeighborLoader
# train_loader = NeighborLoader(data, num_neighbors=[...], batch_size=..., input_nodes=train_mask)


# TODO 4: Define GraphSAGE model
# Use hidden_channels=256, 2 layers, dropout=0.5


# TODO 5: Training loop (mini-batch)
# Track loss and val accuracy per epoch
# Tip: mini-batch training on larger graphs needs more epochs or a learning rate schedule


# TODO 6: Ablation — try different num_neighbors settings
import time

neighbor_configs = [
    [5, 5],
    [15, 10],
    [25, 15]
]

for cfg in neighbor_configs:
    start = time.time()
    # TODO: train with this config and time it
    elapsed = time.time() - start
    print(f'num_neighbors={cfg} | Time: {elapsed:.1f}s | Test acc: ???')

print('Fill in the TODOs above!')

### Hints

<details>
<summary>Hint 1 — Creating masks</summary>

```python
for cls in range(num_classes):
    idx = (data.y == cls).nonzero(as_tuple=True)[0]
    perm = idx[torch.randperm(len(idx))]
    train_mask[perm[:train_per_class]] = True

remaining = (~train_mask).nonzero(as_tuple=True)[0]
perm2 = remaining[torch.randperm(len(remaining))]
val_size = int(len(remaining) * val_ratio)
val_mask[perm2[:val_size]] = True
test_mask[perm2[val_size:]] = True
```
</details>

<details>
<summary>Hint 2 — Timing each config</summary>

```python
start = time.time()
for epoch in range(5):
    for batch in loader:
        ...  # forward + backward
elapsed = time.time() - start
print(f'5 epochs: {elapsed:.2f}s')
```
</details>

---
## What's Next?

In **Notebook 6**, we shift from node-level to **graph-level** tasks — classifying *entire graphs* (e.g., molecules). This introduces global pooling and the **Graph Isomorphism Network (GIN)**, theoretically the most expressive GNN.